In [13]:
import numpy as np
import csv

# County names
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

# Month names
months = ["January", "February", "March", "April",
          "May", "June", "July", "August",
          "September", "October", "November", "December"]

# Clean rainfall data
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70]
]

# Raw Machakos rainfall data
machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

print("Libraries imported and rainfall data prepared successfully.")

Libraries imported and rainfall data prepared successfully.


In [14]:
def clean_machakos(data):
    valid_values = []

    for value in data:
        try:
            valid_values.append(int(value))
        except ValueError:
            pass

    average = round(np.mean(valid_values))

    cleaned_values = []
    imputed_months = []

    for i, value in enumerate(data):
        try:
            cleaned_values.append(int(value))
        except ValueError:
            cleaned_values.append(average)
            imputed_months.append(months[i])

    print("Imputed months:", imputed_months)
    print("Imputed rainfall value:", average, "mm")

    return cleaned_values


machakos_clean = clean_machakos(machakos_raw)

print("Cleaned Machakos data:", machakos_clean)
print("Number of months:", len(machakos_clean))

Imputed months: ['April', 'September']
Imputed rainfall value: 68 mm
Cleaned Machakos data: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]
Number of months: 12


In [15]:
rainfall = np.array(clean_rows + [machakos_clean])

print("Combined rainfall array:")
print(rainfall)

print("Array shape:", rainfall.shape)
print("Array data type:", rainfall.dtype)

Combined rainfall array:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Array shape: (5, 12)
Array data type: int64


In [16]:
# Calculate annual rainfall totals for each county
annual_totals = np.sum(rainfall, axis=1)

print("Annual rainfall totals:")
for county, total in zip(counties, annual_totals):
    print(county, ":", total, "mm")


# Calculate average rainfall for each month
monthly_averages = np.mean(rainfall, axis=0)

print("\nAverage rainfall per month:")
for month, average in zip(months, monthly_averages):
    print(month, ":", round(average, 2), "mm")

Annual rainfall totals:
Nairobi : 1045 mm
Nakuru : 916 mm
Kisumu : 1371 mm
Mombasa : 1075 mm
Machakos : 811 mm

Average rainfall per month:
January : 47.0 mm
February : 47.6 mm
March : 91.0 mm
April : 164.4 mm
May : 171.2 mm
June : 67.6 mm
July : 56.6 mm
August : 58.2 mm
September : 64.2 mm
October : 67.8 mm
November : 127.4 mm
December : 80.6 mm


In [17]:
# Find the wettest month for each county
wettest_indices = np.argmax(rainfall, axis=1)

wettest_months = [months[i] for i in wettest_indices]

print("Wettest month for each county:")

for county, month in zip(counties, wettest_months):
    print(county, ":", month)

Wettest month for each county:
Nairobi : April
Nakuru : April
Kisumu : April
Mombasa : May
Machakos : November


In [18]:
# Create a boolean mask for dry months
dry_month_mask = rainfall < 50

# Count dry months for each county
dry_month_counts = np.sum(dry_month_mask, axis=1)

print("Number of dry months per county:")

for county, count in zip(counties, dry_month_counts):
    print(county, ":", count)

Number of dry months per county:
Nairobi : 4
Nakuru : 3
Kisumu : 0
Mombasa : 2
Machakos : 6


In [19]:
# Calculate rainfall for consecutive months
two_month_rainfall = rainfall[:, :-1] + rainfall[:, 1:]

# Find the starting month of the highest rainfall pair
planting_indices = np.argmax(two_month_rainfall, axis=1)

# Create planting windows
planting_windows = [
    f"{months[i]} - {months[i+1]}"
    for i in planting_indices
]

print("Recommended planting windows:")

for county, window in zip(counties, planting_windows):
    print(county, ":", window)

Recommended planting windows:
Nairobi : April - May
Nakuru : April - May
Kisumu : April - May
Mombasa : April - May
Machakos : November - December


## Planting Advisory

Farmers in Nairobi, Nakuru, Kisumu and Mombasa are advised to consider planting during April and May, which recorded the highest combined rainfall in these counties. Farmers in Machakos are advised to consider November and December as their preferred planting window. The NGO should send county-specific SMS alerts before the recommended planting periods to help farmers prepare their land and seeds. Farmers should also monitor local weather forecasts before planting because rainfall patterns can change. These recommendations are based on historical monthly rainfall readings and should be used alongside local agricultural guidance.

## Data Quality Notes

The original Machakos dataset contained two invalid entries: "N/A" for April and an empty value for September. Both values were replaced with the rounded average of the ten valid Machakos readings, which was 67 mm. This imputation allows the analysis to use a complete dataset, but it introduces uncertainty into the Machakos rainfall totals and planting recommendations.

In [20]:
# Create the rainfall summary CSV file

with open("rainfall_summary.csv", "w", newline="") as file:
    writer = csv.writer(file)

    # Write the required column headings
    writer.writerow([
        "county",
        "annual_total_mm",
        "wettest_month",
        "dry_months",
        "planting_window"
    ])

    # Write the five county results
    for row in zip(
        counties,
        annual_totals,
        wettest_months,
        dry_month_counts,
        planting_windows
    ):
        writer.writerow(row)

print("rainfall_summary.csv created successfully!")

rainfall_summary.csv created successfully!


In [21]:
# Display the contents of the CSV file

with open("rainfall_summary.csv", "r") as file:
    print(file.read())

county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,April,4,April - May
Nakuru,916,April,3,April - May
Kisumu,1371,April,0,April - May
Mombasa,1075,May,2,April - May
Machakos,811,November,6,November - December



In [22]:
%%writefile requirements.txt
numpy

Overwriting requirements.txt


In [23]:
%%writefile README.md
# Rainfall Advisory Analysis

- This project analyses monthly rainfall data for five Kenyan counties using NumPy, including data cleaning, annual totals, monthly averages, dry months and planting windows.
- To run the project, open rainfall_analysis.ipynb in Google Colab or Jupyter Notebook, install the libraries in requirements.txt, and run all cells from top to bottom.
- Planting advisory: Nairobi, Nakuru, Kisumu and Mombasa have an April-May planting window, while Machakos has a November-December planting window; farmers should also consult local weather forecasts.

Overwriting README.md


In [24]:
import os

print("Project files:")
for filename in [
    "README.md",
    "rainfall_analysis.ipynb",
    "rainfall_summary.csv",
    "requirements.txt"
]:
    print(filename, ":", os.path.exists(filename))

Project files:
README.md : True
rainfall_analysis.ipynb : False
rainfall_summary.csv : True
requirements.txt : True
